In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

# Downloading E-mini S&P 500 futures
es = yf.download(
    "ES=F",
    start="2025-07-30",
    end="2026-03-10",
    interval="60m",
    auto_adjust=True
)

es.columns = es.columns.droplevel(1)
es = es.reset_index()
es['Datetime'] = pd.to_datetime(es['Datetime'])
es = es.set_index('Datetime')

overnight = es[
    (es.index.time >= pd.to_datetime("18:00").time()) |
    (es.index.time <= pd.to_datetime("09:29").time())
].copy()

overnight['Return'] = overnight['Close'].pct_change()
overnight['Date'] = overnight.index.date

overnight_features = overnight.groupby('Date').agg(
    Overnight_Return = ('Return', 'sum'),
    Overnight_Volatility = ('Return', 'std'),
    Futures_Last_Price = ('Close', 'last')
)

overnight_features.to_csv("ES_overnight_features.csv")
print(overnight_features.head())

[*********************100%***********************]  1 of 1 completed

            Overnight_Return  Overnight_Volatility  Futures_Last_Price
Date                                                                  
2025-07-30          0.004229              0.002529             6442.50
2025-07-31         -0.012397              0.002761             6362.75
2025-08-01         -0.015511              0.002637             6264.50
2025-08-03          0.001077              0.000085             6271.25
2025-08-04          0.014710              0.001766             6364.00
